In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q vllm
print("install done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.7/303.7 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.8/

In [3]:
# ============================================================================
# PHASE 7: FORCING VARIANTS AT TWO BUDGETS
# One 16k generation -> 8k condition derived by truncation -> 3 forcing variants
# ============================================================================

import time, json, os, re, csv
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset

os.makedirs("outputs", exist_ok=True)
MODEL = "WeiboAI/VibeThinker-3B"
K = 4
GEN_BUDGET = 16384
BUDGETS = [8192, 16384]

V1 = ("\n\n</think>\n\nI have reasoned enough. Based on the work above, "
      "the final answer is \\boxed{")
V2_SUM = ("\n\n</think>\n\nI am out of reasoning time. Let me summarize exactly "
          "what I have established so far.\n\nSummary of established facts:")
V2_ANS = "\n\nTherefore, the final answer is \\boxed{"
V3 = ("\n\n[Note: you have very little space left. Stop exploring and commit to "
      "your best answer now.]\n\n</think>\n\nThe final answer is \\boxed{")

ds = load_dataset("math-ai/aime25")["test"]
tokenizer = AutoTokenizer.from_pretrained(MODEL)

llm = LLM(model=MODEL, dtype="float16", gpu_memory_utilization=0.90,
          max_model_len=18432, trust_remote_code=True)
print("\nModel loaded.\n")


def build_prompt(q):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)


def extract_boxed(t):
    i = t.rfind("\\boxed{")
    if i == -1:
        return None
    j, d, o = i + 7, 1, []
    while j < len(t) and d > 0:
        c = t[j]
        if c == "{":
            d += 1
        elif c == "}":
            d -= 1
            if d == 0:
                break
        o.append(c); j += 1
    return "".join(o).strip()


def to_int(s):
    if s is None:
        return None
    m = re.search(r"-?\d+", s.replace(",", ""))
    return int(m.group()) if m else None


prompts = [build_prompt(ds[i]["problem"]) for i in range(30)]
truths = [to_int(str(ds[i]["answer"])) for i in range(30)]

# ---------------------------------------------------------------- generation
print("=" * 78)
print(f"STAGE 1: generating 30 x K={K} at {GEN_BUDGET:,} tokens")
print("Expect roughly 3-4 hours.")
print("=" * 78)

t0 = time.time()
outs = llm.generate(prompts, SamplingParams(
    n=K, temperature=1.0, top_p=0.95, max_tokens=GEN_BUDGET))
gen_time = time.time() - t0
gen_tokens = sum(len(c.token_ids) for o in outs for c in o.outputs)
print(f"done: {gen_time/60:.1f} min, {gen_tokens:,} tokens, "
      f"{gen_tokens/gen_time:.0f} tok/s\n")

# keep every trace so future variants need no regeneration
traces = [[{"ids": list(c.token_ids), "text": c.text,
            "finish": c.finish_reason} for c in o.outputs] for o in outs]
with open("outputs/phase7_traces.json", "w") as f:
    json.dump({"truths": truths, "traces": traces}, f)
print("traces saved to outputs/phase7_traces.json\n")


def prefix_at(t, budget):
    """(text, was_truncated) for this trace capped at `budget` tokens."""
    if len(t["ids"]) <= budget:
        return t["text"], (t["finish"] != "stop")
    return tokenizer.decode(t["ids"][:budget]), True


GREEDY = SamplingParams(n=1, temperature=0.0, max_tokens=24)
SUMMARY = SamplingParams(n=1, temperature=0.0, max_tokens=320)

results, force_cost = {}, {}

for BUDGET in BUDGETS:
    print("=" * 78)
    print(f"BUDGET = {BUDGET:,}")
    print("=" * 78)

    base, jobs = [], []
    for pi in range(30):
        row = []
        for ci in range(K):
            txt, trunc = prefix_at(traces[pi][ci], BUDGET)
            row.append(to_int(extract_boxed(txt)))
            if trunc:
                jobs.append((pi, ci, prompts[pi] + txt))
        base.append(row)

    n_tr = len(jobs)
    acc0 = sum(1 for pi in range(30) for p in base[pi] if p == truths[pi])
    print(f"  truncated: {n_tr}/{120}")
    print(f"  [no forcing] {100*acc0/120:.1f}%\n")
    results[f"{BUDGET}_none"] = round(100 * acc0 / 120, 1)

    # ---- V1 and V3: single-pass forcing --------------------------------
    for tag, phrase in [("V1_bare", V1), ("V3_deadline", V3)]:
        t0 = time.time()
        fo = llm.generate([j[2] + phrase for j in jobs], GREEDY)
        cost = time.time() - t0
        preds = [r[:] for r in base]
        for (pi, ci, _), o in zip(jobs, fo):
            preds[pi][ci] = to_int(o.outputs[0].text)
        acc = sum(1 for pi in range(30) for p in preds[pi] if p == truths[pi])
        results[f"{BUDGET}_{tag}"] = round(100 * acc / 120, 1)
        force_cost[f"{BUDGET}_{tag}"] = round(cost / 60, 1)
        print(f"  [{tag:12}] {100*acc/120:5.1f}%   "
              f"({100*(acc-acc0)/120:+.1f})   {cost/60:.1f} min")

    # ---- V2: summarize, then answer ------------------------------------
    t0 = time.time()
    sums = llm.generate([j[2] + V2_SUM for j in jobs], SUMMARY)
    fo = llm.generate(
        [j[2] + V2_SUM + s.outputs[0].text + V2_ANS for j, s in zip(jobs, sums)],
        GREEDY)
    cost = time.time() - t0
    preds = [r[:] for r in base]
    ex = []
    for (pi, ci, _), s, o in zip(jobs, sums, fo):
        preds[pi][ci] = to_int(o.outputs[0].text)
        if len(ex) < 6 and pi in (1, 13, 27):
            ex.append({"problem": pi, "truth": truths[pi],
                       "summary": s.outputs[0].text[:400],
                       "answer": o.outputs[0].text[:40],
                       "parsed": preds[pi][ci]})
    acc = sum(1 for pi in range(30) for p in preds[pi] if p == truths[pi])
    results[f"{BUDGET}_V2_summarize"] = round(100 * acc / 120, 1)
    force_cost[f"{BUDGET}_V2_summarize"] = round(cost / 60, 1)
    print(f"  [{'V2_summarize':12}] {100*acc/120:5.1f}%   "
          f"({100*(acc-acc0)/120:+.1f})   {cost/60:.1f} min\n")

    json.dump({"results": results, "force_minutes": force_cost,
               "gen_minutes": round(gen_time / 60, 1), "v2_examples": ex},
              open("outputs/phase7_results.json", "w"), indent=2)

# ------------------------------------------------------------------ summary
print("=" * 78)
print("FINAL RESULTS")
print("=" * 78)
print(f"{'Budget':>8}{'Method':>16}{'Pass@1':>10}{'vs none':>10}")
print("-" * 78)
for B in BUDGETS:
    b0 = results[f"{B}_none"]
    for tag in ["none", "V1_bare", "V3_deadline", "V2_summarize"]:
        v = results[f"{B}_{tag}"]
        d = "" if tag == "none" else f"{v-b0:+.1f}"
        print(f"{B//1024:>7}k{tag:>16}{v:>9.1f}%{d:>10}")
    print("-" * 78)
print()
print("Reference points from earlier phases:")
print("   4k + V1 bare  : 40.8%")
print("   8k + V1 bare  : 54.2%   <- Phase 7 should reproduce this")
print("  32k no forcing : 80.8%")
print()

print("=" * 78)
print("V2 SUMMARY EXAMPLES (is the summary substantive or hand-waving?)")
print("=" * 78)
for e in ex:
    hit = "HIT" if e["parsed"] == e["truth"] else "miss"
    print(f"problem {e['problem']} truth={e['truth']} parsed={e['parsed']} [{hit}]")
    print(f"  summary: {e['summary'][:300]}")
    print(f"  answer : {e['answer']}")
    print()

with open("outputs/phase7_results.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["budget", "method", "pass_at_1", "force_minutes"])
    for B in BUDGETS:
        for tag in ["none", "V1_bare", "V3_deadline", "V2_summarize"]:
            w.writerow([B, tag, results[f"{B}_{tag}"],
                        force_cost.get(f"{B}_{tag}", 0)])
print("Saved outputs/phase7_results.json / .csv / phase7_traces.json")
print(f"TOTAL RUNTIME: {(time.time()-t0)/60:.0f} min")

VersionError: Detected mismatched Protobuf Gencode/Runtime major versions when loading google/protobuf/duration.proto: gencode 6.33.6 runtime 5.29.5. Same major version is required. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.

In [ ]:
# ============================================================================
# PHASE 8: DEEP ANALYSIS FROM SAVED TRACES  (no GPU, ~1 minute)
# ============================================================================
import json, os, re, random
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

os.makedirs("outputs", exist_ok=True)
tok = AutoTokenizer.from_pretrained("WeiboAI/VibeThinker-3B")
d = json.load(open("outputs/phase7_traces.json"))
truths, traces = d["truths"], d["traces"]

def boxed(t):
    i = t.rfind("\\boxed{")
    if i == -1: return None
    j, dep, o = i+7, 1, []
    while j < len(t) and dep > 0:
        c = t[j]
        if c == "{": dep += 1
        elif c == "}":
            dep -= 1
            if dep == 0: break
        o.append(c); j += 1
    return "".join(o).strip()

def as_int(s):
    if s is None: return None
    m = re.search(r"-?\d+", s.replace(",", ""))
    return int(m.group()) if m else None

# ---- fine-grained no-forcing curve (FREE) ---------------------------------
GRID = [1024,2048,3072,4096,6144,8192,10240,12288,14336,16384]
curve, trunc_curve, flat = [], [], []
for B in GRID:
    ok = tr = 0
    for pi in range(30):
        for ci in range(4):
            t = traces[pi][ci]
            if len(t["ids"]) <= B:
                txt, is_tr = t["text"], (t["finish"] != "stop")
            else:
                txt, is_tr = tok.decode(t["ids"][:B]), True
            tr += is_tr
            if as_int(boxed(txt)) == truths[pi]: ok += 1
    curve.append(100*ok/120); trunc_curve.append(100*tr/120)
    flat.append({"budget": B, "acc": round(100*ok/120,1),
                 "trunc_pct": round(100*tr/120,1)})
    print(f"  {B:>6,} tokens -> {100*ok/120:5.1f}%   truncated {100*tr/120:5.1f}%")

# ---- bootstrap CI on the forcing gain -------------------------------------
def boot_ci(gain_pts, n=120, iters=4000):
    """95% CI for a difference of proportions via bootstrap."""
    random.seed(0)
    k = round(gain_pts/100*n)
    vals = [1]*k + [0]*(n-k)
    means = []
    for _ in range(iters):
        s = [random.choice(vals) for _ in range(n)]
        means.append(100*sum(s)/n)
    means.sort()
    return means[int(.025*iters)], means[int(.975*iters)]

print()
print("="*74)
print("FORCING GAIN WITH 95% BOOTSTRAP CI")
print("="*74)
GAINS = {4096:(27.5,40.8,104), 8192:(46.7,56.7,81), 16384:(63.3,70.8,58)}
ci_rows = []
for B,(b,f,ntr) in GAINS.items():
    lo, hi = boot_ci(f-b)
    sig = "yes" if lo > 0 else "NO"
    print(f"  {B//1024:>3}k  {b:5.1f}% -> {f:5.1f}%   gain {f-b:+5.1f} "
          f"[{lo:+.1f},{hi:+.1f}]  significant={sig}   recovery={100*(f-b)/100*120/ntr:5.1f}%")
    ci_rows.append({"budget":B,"base":b,"forced":f,"gain":round(f-b,1),
                    "ci_lo":round(lo,1),"ci_hi":round(hi,1),
                    "recovery_pct":round((f-b)/100*120/ntr*100,1)})

# ---- Figure: main result ---------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2,4.6), dpi=150)
ax.plot(GRID, curve, "-", color="#B23A3A", lw=2, label="No forcing (fine grid)")
fb = [4096,8192,16384]; fa = [40.8,56.7,70.8]
ax.plot(fb, fa, "s-", color="#1F6E43", lw=2, ms=9, label="Best forcing")
ax.plot([32768],[80.8],"*",color="#2F4B7C",ms=18,label="32k unforced (baseline)")
ax.axhline(91.4, ls=":", color="gray", lw=1.4)
ax.text(1100, 92.4, "paper: 91.4%", fontsize=8, color="gray")
for x,y in zip(fb,fa):
    ax.annotate(f"{y:.1f}",(x,y),textcoords="offset points",xytext=(0,9),
                ha="center",fontsize=8,color="#1F6E43")
ax.set_xscale("log", base=2)
ax.set_xticks(fb+[16384,32768]); ax.set_xticklabels(["4k","8k","16k","16k","32k"])
ax.set_xlabel("Token budget per sample"); ax.set_ylabel("AIME25 Pass@1 (%)")
ax.set_title("Reasoning budget, not reasoning ability, bounds accuracy")
ax.set_ylim(15,100); ax.grid(alpha=.3); ax.legend(fontsize=9, loc="lower right")
plt.tight_layout(); plt.savefig("outputs/fig3_main_result.png", bbox_inches="tight")

# ---- Figure: truncation drives everything ----------------------------------
fig, ax1 = plt.subplots(figsize=(7.2,4.6), dpi=150)
ax1.plot(GRID, trunc_curve, "-", color="#C46A1F", lw=2)
ax1.set_xscale("log", base=2); ax1.set_xticks(GRID[::2])
ax1.set_xticklabels([f"{g//1024}k" for g in GRID[::2]])
ax1.set_xlabel("Token budget"); ax1.set_ylabel("% samples truncated", color="#C46A1F")
ax1.tick_params(axis="y", labelcolor="#C46A1F"); ax1.grid(alpha=.3)
ax2 = ax1.twinx()
ax2.plot(GRID, curve, "-", color="#B23A3A", lw=2)
ax2.set_ylabel("Pass@1 (%)", color="#B23A3A"); ax2.tick_params(axis="y", labelcolor="#B23A3A")
ax1.set_title("Truncation rate mirrors accuracy exactly")
plt.tight_layout(); plt.savefig("outputs/fig4_truncation.png", bbox_inches="tight")

# ---- Figure: length histogram ----------------------------------------------
lens = [len(traces[p][c]["ids"]) for p in range(30) for c in range(4)]
fig, ax = plt.subplots(figsize=(7.2,4), dpi=150)
ax.hist(lens, bins=30, color="#2F4B7C", edgecolor="white")
ax.axvline(16384, color="#B23A3A", ls="--", lw=2)
ax.text(15200, ax.get_ylim()[1]*.85, "16k cap", rotation=90, fontsize=8, color="#B23A3A")
ax.set_xlabel("Trace length (tokens)"); ax.set_ylabel("Number of samples")
ax.set_title("Reasoning-length distribution (30 problems x K=4)")
plt.tight_layout(); plt.savefig("outputs/fig5_lengths.png", bbox_inches="tight")

json.dump({"fine_curve": flat, "forcing_ci": ci_rows,
           "lengths": {"median": sorted(lens)[len(lens)//2],
                       "at_cap": sum(1 for L in lens if L >= 16384)}},
          open("outputs/phase8_analysis.json","w"), indent=2)
print("\nSaved fig3/fig4/fig5 .png + phase8_analysis.json")

In [ ]:
# ============================================================================
# PHASE 9: PAIRED ANALYSIS - does forcing ever HURT? + hybrid strategy
# ============================================================================
import json, os, re, time, math
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset

os.makedirs("outputs", exist_ok=True)
MODEL = "WeiboAI/VibeThinker-3B"
V1 = ("\n\n</think>\n\nI have reasoned enough. Based on the work above, "
      "the final answer is \\boxed{")

d = json.load(open("outputs/phase7_traces.json"))
truths, traces = d["truths"], d["traces"]
ds = load_dataset("math-ai/aime25")["test"]
tok = AutoTokenizer.from_pretrained(MODEL)
llm = LLM(model=MODEL, dtype="float16", gpu_memory_utilization=0.90,
          max_model_len=18432, trust_remote_code=True)

def prompt_of(i):
    return tok.apply_chat_template([{"role":"user","content":ds[i]["problem"]}],
                                   tokenize=False, add_generation_prompt=True)

def boxed(t):
    i = t.rfind("\\boxed{")
    if i == -1: return None
    j, dep, o = i+7, 1, []
    while j < len(t) and dep > 0:
        c = t[j]
        if c == "{": dep += 1
        elif c == "}":
            dep -= 1
            if dep == 0: break
        o.append(c); j += 1
    return "".join(o).strip()

def as_int(s):
    if s is None: return None
    m = re.search(r"-?\d+", s.replace(",",""))
    return int(m.group()) if m else None

GREEDY = SamplingParams(n=1, temperature=0.0, max_tokens=24)
report = {}

for BUDGET in [8192, 16384]:
    print("="*74); print(f"BUDGET {BUDGET:,}"); print("="*74)
    jobs, base = [], []
    for pi in range(30):
        row = []
        for ci in range(4):
            t = traces[pi][ci]
            if len(t["ids"]) <= BUDGET:
                txt, trunc = t["text"], (t["finish"] != "stop")
            else:
                txt, trunc = tok.decode(t["ids"][:BUDGET]), True
            pred = as_int(boxed(txt))
            row.append({"pred": pred, "trunc": trunc,
                        "had_boxed": pred is not None})
            if trunc:
                jobs.append((pi, ci, prompt_of(pi) + txt))
        base.append(row)

    t0 = time.time()
    fo = llm.generate([j[2] + V1 for j in jobs], GREEDY)
    print(f"  forcing took {(time.time()-t0)/60:.1f} min\n")

    # ---- paired contingency -------------------------------------------
    b = c = same_r = same_w = 0
    recs, forced_map = [], {}
    for (pi, ci, _), o in zip(jobs, fo):
        old = base[pi][ci]["pred"]
        new = as_int(o.outputs[0].text)
        forced_map[(pi, ci)] = new
        o_ok, n_ok = (old == truths[pi]), (new == truths[pi])
        if not o_ok and n_ok: b += 1
        elif o_ok and not n_ok: c += 1
        elif o_ok and n_ok: same_r += 1
        else: same_w += 1
        recs.append({"problem": pi, "sample": ci, "truth": truths[pi],
                     "had_boxed": base[pi][ci]["had_boxed"],
                     "before": old, "after": new,
                     "before_ok": o_ok, "after_ok": n_ok})

    n_disc = b + c
    mcnemar = ((abs(b-c)-1)**2)/n_disc if n_disc > 0 else 0.0
    # p-value from chi2 with 1 df
    p = math.erfc(math.sqrt(mcnemar/2)) if mcnemar > 0 else 1.0

    # ---- three scoring strategies -------------------------------------
    acc_none = sum(1 for pi in range(30) for ci in range(4)
                   if base[pi][ci]["pred"] == truths[pi])
    acc_force = sum(1 for pi in range(30) for ci in range(4)
                    if forced_map.get((pi,ci), base[pi][ci]["pred"]) == truths[pi])
    acc_hyb = 0
    for pi in range(30):
        for ci in range(4):
            cell = base[pi][ci]
            v = cell["pred"] if cell["had_boxed"] else forced_map.get((pi,ci))
            acc_hyb += (v == truths[pi])

    print(f"  truncated samples        : {len(jobs)}")
    print(f"  of those, had a boxed ans: {sum(1 for r in recs if r['had_boxed'])}")
    print()
    print("  PAIRED CONTINGENCY (truncated samples only)")
    print(f"    wrong -> RIGHT  (rescued)  b = {b}")
    print(f"    right -> WRONG  (damaged)  c = {c}")
    print(f"    right -> right             {same_r}")
    print(f"    wrong -> wrong             {same_w}")
    print(f"    McNemar chi2 = {mcnemar:.2f}   p = {p:.4f}"
          f"   {'SIGNIFICANT' if p < 0.05 else 'not significant'}")
    print()
    print("  STRATEGY COMPARISON")
    print(f"    A no forcing        : {100*acc_none/120:5.1f}%")
    print(f"    B force everything  : {100*acc_force/120:5.1f}%")
    print(f"    C HYBRID (keep box) : {100*acc_hyb/120:5.1f}%"
          f"   {'<-- BEST' if acc_hyb >= max(acc_none, acc_force) else ''}")
    print()

    report[BUDGET] = {"n_truncated": len(jobs), "b_rescued": b, "c_damaged": c,
                      "mcnemar_chi2": round(mcnemar,2), "p_value": round(p,4),
                      "acc_none": round(100*acc_none/120,1),
                      "acc_force": round(100*acc_force/120,1),
                      "acc_hybrid": round(100*acc_hyb/120,1),
                      "per_sample": recs}
    json.dump(report, open("outputs/phase9_paired.json","w"), indent=2)

print("="*74)
print("SUMMARY")
print("="*74)
print(f"{'Budget':>8}{'None':>9}{'Force':>9}{'Hybrid':>9}"
      f"{'rescued':>10}{'damaged':>10}{'p':>10}")
for B, r in report.items():
    print(f"{B//1024:>7}k{r['acc_none']:>8.1f}%{r['acc_force']:>8.1f}%"
          f"{r['acc_hybrid']:>8.1f}%{r['b_rescued']:>10}{r['c_damaged']:>10}"
          f"{r['p_value']:>10.4f}")
print("="*74)
print("Saved outputs/phase9_paired.json")